# Stage 4: Grounded answers with Ollama

This notebook completes the RAG loop:

1. Retrieve relevant chunks from Chroma  
2. Build a prompt with that **context** + the **question**  
3. Ask a local Ollama **chat** model to answer **only** from the context  
4. If the context is weak or unrelated, the model (and a simple score check) should say the documents do not contain enough information

Still no CLI app — that is Stage 5.

## Grounded generation (concept)

```
Question → embed → Chroma top-k chunks → prompt(context, question) → Ollama → Answer
```

**Grounded** means: the answer should come from the retrieved text, not from the model inventing Harbor & Co. policy.

We use two safety layers beginners can understand:

1. **Prompt rules** — tell the model to refuse when context is insufficient  
2. **Similarity gate** — if the best chunk is too far from the question, skip the LLM and return a fixed "not enough information" message

**Prerequisites:** Ollama running; `nomic-embed-text`; a small chat model such as `qwen2.5:0.5b` (default) or `llama3.2:3b`.

Also run Stage 3 once (or let this notebook rebuild the collection if empty).

## Paths, models, and helpers

In [ ]:
from pathlib import Path

import chromadb
import requests

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "knowledge_base").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

KB_DIR = PROJECT_ROOT / "data" / "knowledge_base"
CHROMA_DIR = PROJECT_ROOT / ".chroma"
COLLECTION_NAME = "harbor_knowledge"

OLLAMA_URL = "http://127.0.0.1:11434"
EMBED_MODEL = "nomic-embed-text"
# Small local chat model (swap to "llama3.2:3b" if you prefer stronger answers)
CHAT_MODEL = "llama3.2:3b"  # lighter option: "qwen2.5:0.5b"

TOP_K = 3
# Cosine distance from Chroma; larger distance = weaker match. Gate refusals above this.
MAX_DISTANCE = 0.32

NOT_ENOUGH_INFO = (
    "I do not have enough information in the Harbor & Co. documents to answer that."
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Chat model: {CHAT_MODEL}")
print(f"Similarity gate: refuse if best distance > {MAX_DISTANCE}")

In [ ]:
def list_ollama_models() -> set[str]:
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    r.raise_for_status()
    return {m["name"] for m in r.json().get("models", [])}


def require_models() -> None:
    try:
        names = list_ollama_models()
    except requests.RequestException as exc:
        raise RuntimeError("Cannot reach Ollama. Start it with: ollama serve") from exc

    def has(prefix: str) -> bool:
        return any(n.startswith(prefix) for n in names)

    if not has(EMBED_MODEL):
        raise RuntimeError(f"Missing embedding model. Run: ollama pull {EMBED_MODEL}")
    if not has(CHAT_MODEL):
        raise RuntimeError(f"Missing chat model. Run: ollama pull {CHAT_MODEL}")
    print("Ollama OK:", ", ".join(sorted(names)))


def embed_text(text: str) -> list[float]:
    response = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": EMBED_MODEL, "prompt": text},
        timeout=120,
    )
    response.raise_for_status()
    vector = response.json()["embedding"]
    if not vector:
        raise RuntimeError("Empty embedding from Ollama")
    return vector


require_models()

## Ensure the Chroma collection exists

If Stage 3 already built `.chroma/`, we reuse it. Otherwise we rebuild from `data/knowledge_base/`.

In [ ]:
def load_text_documents(folder: Path) -> list[dict]:
    return [
        {"source": path.name, "text": path.read_text(encoding="utf-8").strip()}
        for path in sorted(folder.glob("*.txt"))
    ]


def chunk_by_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in text.split("\n\n") if p.strip()]


def build_chunk_records(documents: list[dict]) -> list[dict]:
    records = []
    for doc in documents:
        for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
            records.append({"source": doc["source"], "chunk_id": i, "text": chunk})
    return records


def rebuild_collection(client):
    chunks = build_chunk_records(load_text_documents(KB_DIR))
    existing = {str(c) for c in client.list_collections()}
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    ids, documents, metadatas, embeddings = [], [], [], []
    for chunk in chunks:
        ids.append(f"{chunk['source']}::{chunk['chunk_id']}")
        documents.append(chunk["text"])
        metadatas.append({"source": chunk["source"], "chunk_id": chunk["chunk_id"]})
        embeddings.append(embed_text(chunk["text"]))
    collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
    print(f"Rebuilt collection with {collection.count()} chunks")
    return collection


CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
names = {str(c) for c in client.list_collections()}

if COLLECTION_NAME in names:
    collection = client.get_collection(COLLECTION_NAME)
    if collection.count() == 0:
        collection = rebuild_collection(client)
    else:
        print(f"Using existing collection with {collection.count()} chunks")
else:
    collection = rebuild_collection(client)

## Retrieve context

In [ ]:
def retrieve(question: str, top_k: int = TOP_K) -> list[dict]:
    """Return a list of {text, source, chunk_id, distance, similarity}."""
    result = collection.query(
        query_embeddings=[embed_text(question)],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc, meta, dist in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append(
            {
                "text": doc,
                "source": meta["source"],
                "chunk_id": meta["chunk_id"],
                "distance": float(dist),
                "similarity": 1.0 - float(dist),
            }
        )
    return hits


def format_context(hits: list[dict]) -> str:
    blocks = []
    for i, hit in enumerate(hits, start=1):
        blocks.append(
            f"[{i}] ({hit['source']}#{hit['chunk_id']})\n{hit['text']}"
        )
    return "\n\n".join(blocks)

## Ask Ollama with a grounded prompt

In [ ]:
SYSTEM_PROMPT = """You are a helpful Harbor & Co. support assistant.
Answer ONLY using the CONTEXT provided by the user.
If the CONTEXT answers the question — including when the answer is "no" or "not allowed" — say that clearly.
Only if the CONTEXT is about a different topic and does not help, say exactly:
I do not have enough information in the Harbor & Co. documents to answer that.
Do not invent policies, products, prices, or dates.
Do not reverse the meaning of the CONTEXT.
Keep answers short and clear."""


def ask_ollama(question: str, context: str) -> str:
    user_content = (
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{question}\n\n"
        "Answer based only on the CONTEXT."
    )
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": CHAT_MODEL,
            "stream": False,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
        },
        timeout=300,
    )
    response.raise_for_status()
    return response.json()["message"]["content"].strip()


def answer_question(question: str) -> dict:
    """Full RAG step: retrieve → gate → generate."""
    hits = retrieve(question)
    best_distance = hits[0]["distance"] if hits else 999.0

    if best_distance > MAX_DISTANCE:
        return {
            "question": question,
            "hits": hits,
            "gated": True,
            "answer": NOT_ENOUGH_INFO,
        }

    context = format_context(hits)
    answer = ask_ollama(question, context)
    return {
        "question": question,
        "hits": hits,
        "gated": False,
        "answer": answer,
    }


def show_answer(question: str) -> dict:
    result = answer_question(question)
    print(f"Question: {question}")
    print("Retrieved:")
    for hit in result["hits"]:
        print(
            f"  - {hit['source']}#{hit['chunk_id']}  "
            f"distance={hit['distance']:.3f}  similarity={hit['similarity']:.3f}"
        )
    print(f"Gated (weak retrieval): {result['gated']}")
    print(f"Answer:\n{result['answer']}\n")
    print("-" * 60)
    return result

## Demo: in-scope questions (should use the documents)

In [ ]:
refund_result = show_answer(
    "How many business days does a refund take to appear on my statement?"
)
shipping_result = show_answer(
    "How long is express delivery after an order leaves the warehouse?"
)

## Demo: out-of-scope question (should refuse)

Either the **distance gate** trips, or the model follows the prompt and says it does not have enough information.

In [ ]:
oos_result = show_answer("What is the capital of France?")

refusal_markers = (
    "do not have enough information",
    "don't have enough information",
    "not enough information",
    "documents to answer",
)
oos_text = oos_result["answer"].lower()
refused = oos_result["gated"] or any(m in oos_text for m in refusal_markers)

print("Checks:")
print(f"  refund gated? {refund_result['gated']} (expect False)")
print(f"  shipping gated? {shipping_result['gated']} (expect False)")
print(f"  out-of-scope refused? {refused} (expect True)")
print(f"  out-of-scope gated? {oos_result['gated']}")

if (not refund_result["gated"]) and (not shipping_result["gated"]) and refused:
    print("OK: in-scope questions generated; out-of-scope refused.")
else:
    print("Unexpected behavior — inspect answers and MAX_DISTANCE.")

## What you should notice

- Retrieval happens **before** generation.
- The prompt tells the model to stay inside the CONTEXT.
- A simple distance gate stops clearly unrelated questions without relying only on the model.
- Small models (`qwen2.5:0.5b`) are fast and local; try `llama3.2:3b` if answers feel too thin.

## Next stage (not built yet)

**Stage 5:** move this working flow into reusable Python scripts and a simple terminal chatbot loop.